# GTU Mimari Lejant - Colab Training

Bu notebook Roboflow YOLOv11 instance segmentation export'u ile ilk YOLO segmentation baseline modelini egitir.

Colab ayari: `Runtime -> Change runtime type -> GPU`. GPU onceligi: A100 > L4 > T4.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Repo'yu Klonla

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r /content/lejanter_doga_vlm_codex/requirements.txt

## 2. Roboflow Zip'i Yukle veya Drive'dan Kopyala

Asagidaki seceneklerden sadece birini calistir:

1. Bilgisayardan upload.
2. Kendi MyDrive path'inden kopyalama.
3. Paylasilan Drive klasorunden `dataset/` ve `weights/` klasorlerini indirme.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print('Uploaded:', zip_name)

## 2B. Google Drive'dan Zip Kopyala

Upload yerine dataset zip dosyasini Google Drive'dan almak istersen bu hucreyi calistir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -f '/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/GTU_MIMARI_LEJANT.yolov11.zip' '/content/GTU_MIMARI_LEJANT.yolov11.zip'
zip_name = '/content/GTU_MIMARI_LEJANT.yolov11.zip'
print('Dataset zip:', zip_name)

## 2C. Paylasilan Drive Klasorunden Dataset ve Weights Al

Paylasilan Drive klasorunde `dataset/` ve `weights/` alt klasorleri varsa bu hucreyi kullan. `dataset/` icindeki zip otomatik bulunur; `weights/` icindeki `.pt` dosyalari da Colab icine kopyalanir.

In [ ]:
from pathlib import Path
import shutil

TEST_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1QJD49ylJs9PpidPDQEcCMhx0RltWUfTT?usp=sharing'
WEIGHTS_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
TEST_DOWNLOAD_DIR = Path('/content/shared_drive_test')
WEIGHTS_DOWNLOAD_DIR = Path('/content/shared_drive_weights')
for folder in [TEST_DOWNLOAD_DIR, WEIGHTS_DOWNLOAD_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

!pip install -q gdown
!gdown --folder '{TEST_DRIVE_FOLDER_URL}' -O /content/shared_drive_test --remaining-ok
!gdown --folder '{WEIGHTS_DRIVE_FOLDER_URL}' -O /content/shared_drive_weights --remaining-ok

print('Downloaded test files:')
for path in sorted(TEST_DOWNLOAD_DIR.rglob('*')):
    print(path)
print('Downloaded weight files:')
for path in sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*')):
    print(path)

dataset_zips = []
for root in [TEST_DOWNLOAD_DIR, WEIGHTS_DOWNLOAD_DIR]:
    dataset_zips.extend(list((root / 'dataset').rglob('*.zip')) if (root / 'dataset').exists() else list(root.rglob('*.zip')))
if dataset_zips:
    zip_name = str(dataset_zips[0])
    print('Dataset zip:', zip_name)
else:
    zip_name = None
    print('No dataset zip found. This is OK if you only want to use Drive weights/test images. Run upload or 2B before dataset extraction if training/evaluation split is needed.')

weights_dst = Path('/content/lejanter_doga_vlm_codex/drive_weights')
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)

weight_files = sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt'))
for pt in weight_files:
    dst = weights_dst / pt.name
    if dst.exists():
        dst = weights_dst / f'{pt.parent.name}-{pt.name}'
    shutil.copy2(pt, dst)
print('Copied weights to:', weights_dst)
for pt in sorted(weights_dst.rglob('*.pt')):
    print('-', pt)
if not weight_files:
    print('No .pt weights found under weights Drive download. Check that WEIGHTS_DRIVE_FOLDER_URL is shared.')

test_src = TEST_DOWNLOAD_DIR / 'test'
if not test_src.exists():
    test_src = TEST_DOWNLOAD_DIR
test_dst = Path('/content/lejanter_doga_vlm_codex/data/raw/drive_test')
if test_dst.exists():
    shutil.rmtree(test_dst)
test_dst.mkdir(parents=True, exist_ok=True)
if test_src.exists():
    for image in test_src.rglob('*'):
        if image.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.avif'}:
            shutil.copy2(image, test_dst / image.name)
    print('Copied test images to:', test_dst)
    for image in sorted(test_dst.glob('*')):
        print('-', image)
else:
    print('No test folder found in test Drive download')

reports_drive_dir = TEST_DOWNLOAD_DIR / 'reports'
print('Downloaded reports folder candidate:', reports_drive_dir)